# [과제] MCP vs RAG 구조 비교 실습 — PydanticAI 기반

이 실습에서는 **같은 질문**에 대해 MCP(Tool 호출) 방식과 RAG(문서 검색) 방식을 각각 구현하고,
두 결과를 비교하여 각 아키텍처의 장단점을 직접 체험합니다.

**목표:**
1. MCP 방식: PydanticAI Tool 1개를 직접 정의하고 Agent가 자동 호출
2. RAG 방식: sample_docs/ 문서를 임베딩하여 유사도 검색 + PydanticAI Agent 답변
3. 같은 질문에 두 방식을 적용하고 장단점 비교

## 환경 설정

필요한 라이브러리를 임포트하고 API 키를 로드합니다.

> **참고**: Jupyter 노트북에서는 `run_sync()` 대신 `await agent.run()`을 사용해야 합니다.
> 노트북은 이미 이벤트 루프가 실행 중이라 `run_sync()`가 충돌합니다.

In [ ]:
from pathlib import Path

import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
from pydantic_ai import Agent

# .env 파일에서 OPENAI_API_KEY 로드
load_dotenv()

# 임베딩 API 호출용 클라이언트
openai_client = OpenAI()

## Part 1: MCP(Tool 호출) 방식

MCP 방식에서는 Agent가 질문을 분석하여 **적절한 Tool을 자동 선택하고 호출**합니다.
여기서는 제품 보증 정보를 반환하는 mock Tool을 만들어 봅니다.

### TODO 1-1: Agent 생성 + Mock Tool 함수 구현

- `Agent("openai:gpt-5.4", instructions=...)`로 Agent 생성
- `@agent.tool_plain` 데코레이터로 Tool 등록
- 함수명: `get_product_warranty(product_name: str) -> str`
- 제품명을 받아 보증 정보를 반환하는 mock 함수
- 최소 2개 제품에 대한 mock 데이터 포함

> **힌트**: docstring이 LLM의 Tool 선택 기준입니다. 어떤 상황에서 이 Tool을 쓸지 명확히 적으세요.

In [ ]:
# Agent 생성 — instructions에 역할을 정의
mcp_agent = Agent(
    "openai:gpt-5.4",
    instructions="당신은 TechShop의 고객 상담 에이전트입니다. Tool을 활용하여 정확한 정보를 제공하세요.",
)


# @tool_plain 데코레이터로 Tool 등록 — docstring이 LLM의 선택 기준
@mcp_agent.tool_plain
def get_product_warranty(product_name: str) -> str:
    """제품 보증 정보를 조회한다. 보증 기간, 보증 범위, 제외 항목을 반환한다.
    고객이 보증, AS, 수리에 대해 질문할 때 사용한다."""
    pass  # TODO: mock 데이터(dict)를 만들고, product_name으로 조회하여 반환하세요

### TODO 1-2: MCP 에이전트 실행 함수

- `async def`로 정의 (내부에서 `await agent.run()` 사용)
- 결과에서 `result.output`을 반환

> **주의**: Jupyter에서는 `run_sync()` 대신 `await agent.run()`을 사용합니다.

In [ ]:
# async def — 내부에서 await를 쓰기 위해 비동기 함수로 정의
async def run_mcp(question: str) -> str:
    """MCP 방식으로 질문에 답변한다."""
    # TODO: await mcp_agent.run(question)을 호출하고 result.output을 반환하세요
    pass

## Part 2: RAG 방식

RAG 방식에서는 문서를 임베딩하여 벡터로 변환한 뒤, 질문과 유사한 문서를 검색하고 그 결과를 LLM에게 제공합니다.
아래 4개 함수를 순서대로 구현하세요.

In [ ]:
# 검색 대상 문서 디렉토리
DOCS_DIR = Path("sample_docs")

### TODO 2-1: 문서 로드 + 청킹

- `sample_docs/` 폴더의 `.md` 파일을 읽어온다
- 단락(`\n\n`) 기준으로 청크를 분할한다
- 각 청크에 출처(filename)를 함께 저장한다

> **힌트**: `Path.glob("*.md")`로 파일 목록을 얻고, `content.split("\n\n")`으로 단락 분리

In [ ]:
def load_and_chunk() -> list[dict]:
    """문서를 로드하고 청크로 분할한다. 각 청크는 {"text": ..., "source": ...} 형태."""
    chunks = []
    # TODO: DOCS_DIR의 .md 파일을 읽고 단락 기준으로 분할하세요
    return chunks

### TODO 2-2: 임베딩 함수

- OpenAI `text-embedding-3-small` 모델 사용
- 텍스트 리스트를 받아 numpy 배열로 반환

> **힌트**: `openai_client.embeddings.create(model=..., input=texts)`로 API 호출

In [ ]:
def embed(texts: list[str]) -> np.ndarray:
    """텍스트를 임베딩 벡터로 변환한다. (text-embedding-3-small 사용)"""
    pass  # TODO: OpenAI Embedding API를 호출하고 numpy 배열로 반환하세요

### TODO 2-3: 유사도 검색 함수

- 코사인 유사도로 질문과 가장 관련 있는 청크 `top_k`개 반환
- 코사인 유사도 = 정규화된 벡터의 내적

> **힌트**: `np.linalg.norm()`으로 정규화 → `@`(행렬 곱)으로 내적 → `np.argsort()`로 정렬

In [ ]:
def search(query: str, chunks: list[dict], chunk_embeddings: np.ndarray, top_k: int = 3) -> list[dict]:
    """질문과 유사한 청크를 코사인 유사도로 검색한다."""
    pass  # TODO: 질문 임베딩 → 코사인 유사도 계산 → 상위 top_k개 반환

### TODO 2-4: RAG 파이프라인 실행 함수

전체 흐름: 문서 로드 → 청킹 → 임베딩 → 검색 → PydanticAI Agent 답변
- 검색된 문서를 Agent의 instructions에 포함하여 답변 생성
- `async def`로 정의해야 합니다 (내부에서 `await agent.run()` 사용)

> **힌트**: 검색 결과를 `[참조 문서]` 형태로 instructions에 넣고, "문서에 없는 내용은 답하지 마라" 규칙을 추가

In [ ]:
# async def — 내부에서 await agent.run()을 사용하므로 비동기 함수로 정의
async def run_rag(question: str) -> str:
    """RAG 방식으로 질문에 답변한다."""
    # TODO: load_and_chunk() → embed() → search() → Agent 생성 → await agent.run()
    pass

## Part 3: 비교 실행

같은 질문을 MCP와 RAG 두 방식으로 실행하여 결과를 비교합니다.
답변의 정확도, 근거 유무, 응답 속도 등을 관찰하세요.

In [ ]:
# 비교할 질문 — MCP와 RAG 모두 이 질문으로 실행
question = "스마트 워치 Pro의 보증 기간이 어떻게 되나요?"

### MCP 방식 실행

Tool 기반으로 답변을 생성합니다. `get_product_warranty` Tool이 자동 호출되는지 확인하세요.

In [ ]:
# run_mcp는 async 함수이므로 await로 호출
mcp_answer = await run_mcp(question)
print(f"\n답변: {mcp_answer}")

### RAG 방식 실행

문서 검색 기반으로 답변을 생성합니다. 어떤 청크가 검색되었는지, 유사도 점수를 비교하세요.

In [ ]:
# run_rag는 async 함수이므로 await로 호출
rag_answer = await run_rag(question)
print(f"\n답변: {rag_answer}")

## Part 4: 장단점 비교

위 실행 결과를 바탕으로, 아래 항목에 대해 자신의 분석을 작성하세요.
MCP와 RAG의 답변을 비교하며, 어떤 차이가 있었는지 구체적으로 기술해 보세요.

### MCP 방식의 장점
- 
- 
- 

### MCP 방식의 단점
- 
- 
- 

### RAG 방식의 장점
- 
- 
- 

### RAG 방식의 단점
- 
- 
- 

### 어떤 상황에서 MCP가 더 적합한가?
- 

### 어떤 상황에서 RAG가 더 적합한가?
- 